### Test Thermal Balance Corrector

Validation suite for the `ThermalBalanceCorrector` class.

This module verifies the physical consistency and mathematical correctness 
of thermal detailed balance enforcement on relaxation superoperators. 
Specifically, it tests three fundamental invariants required for valid 
open quantum system dynamics:

1. Detailed Balance Enforcement
   Confirms that all population transition rates satisfy the Boltzmann 
   ratio: w_{j→i} / w_{i→j} = exp((E_j - E_i) / k_B T).

2. Trace Preservation (Probability Conservation)
   Verifies that the column sums of the population block, combined with 
   the updated diagonal elements, remain exactly zero. This ensures 
   d(Tr[ρ])/dt = 0

3. Dephasing Rate Correction
   Validates that coherence dephasing rates (off-diagonal density matrix 
   elements) are updated according to the relation:
   ΔΓ_{ij} = 0.5 * (Δγ_i_out + Δγ_j_out), where Δγ represents the 
   change in population out-rates induced by thermal scaling.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

import torch
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from mars.population.thermal_corrections import ThermalBalanceCorrector


# ==============================================================================
# TEST 1: DETAILED BALANCE ENFORCEMENT
# ==============================================================================
def test_detailed_balance():
    """
    Verifies that w_{j→i} / w_{i→j} = exp((E_j - E_i) / k_B T)
    for all population transitions after thermal transformation.
    """
    N = 4
    energies = torch.tensor([0.0, 1.5, 3.0, 5.0])
    T = torch.tensor(2.0)
    corrector = ThermalBalanceCorrector(energies, thermal_balance_mode="symmetric")
    
    # Create dummy symmetric mean strengths k'_ij
    torch.manual_seed(42)
    k_prime = torch.rand(N, N) * 0.5
    k_prime = 0.5 * (k_prime + k_prime.T)  # Symmetrize
    k_prime.fill_diagonal_(0.0)
    
    # Build valid initial superoperator
    N2 = N * N
    superop = torch.zeros(N2, N2)
    pop_idx = torch.arange(N) * (N + 1)
    superop[pop_idx[:, None], pop_idx[None, :]] = k_prime
    # Initial diagonal: negative out-rates
    superop[pop_idx, pop_idx] = -k_prime.sum(dim=-2)
    
    # Apply thermal transform
    transformed = corrector.apply_superoperator_transform(T, superop)
    new_pop = transformed[pop_idx[:, None], pop_idx[None, :]]
    
    # Check detailed balance ratio
    ratio = new_pop / new_pop.T
    expected_ratio = torch.exp(corrector.omega_K / T)
    
    # Mask diagonals and numerically zero entries to avoid division artifacts
    mask = ~torch.eye(N, dtype=torch.bool) & (new_pop > 1e-8)
    
    assert torch.allclose(ratio[mask], expected_ratio[mask], rtol=1e-4), \
        f"Detailed balance violated. Max relative error: {(ratio[mask] - expected_ratio[mask]).abs().max()}"
    print("✅ Test 1 Passed: Detailed balance correctly enforced.")


# ==============================================================================
# TEST 2: TRACE PRESERVATION (PROBABILITY CONSERVATION)
# ==============================================================================
def test_trace_preservation():
    """
    Verifies that column sums of the population block + corresponding 
    superoperator diagonal equal zero: Σ_i R_{i,j} + R_{j,j} = 0.
    This ensures d(Tr[ρ])/dt = 0.
    """
    N = 3
    energies = torch.tensor([0.0, 2.0, 4.5])
    T = torch.tensor(1.2)
    corrector = ThermalBalanceCorrector(energies, thermal_balance_mode="symmetric")
    
    torch.manual_seed(99)
    # Random valid rate matrix
    rates = torch.rand(N, N) * 0.4
    rates = 0.5 * (rates + rates.T)
    rates.fill_diagonal_(0.0)
    
    N2 = N * N
    superop = torch.zeros(N2, N2)
    pop_idx = torch.arange(N) * (N + 1)
    superop[pop_idx[:, None], pop_idx[None, :]] = rates
    superop[pop_idx, pop_idx] = -rates.sum(dim=-2)  # Valid initial trace-preserving diagonal
    
    # Apply transform
    transformed = corrector.apply_superoperator_transform(T, superop)
    
    # Extract population columns (all rows, only population columns)
    pop_columns = transformed[:, pop_idx]
    
    # Column sums must be exactly zero for trace preservation
    col_sums = pop_columns.sum(dim=0)
    
    assert torch.allclose(col_sums, torch.zeros_like(col_sums), atol=1e-6), \
        f"Trace preservation broken! Column sums: {col_sums}"
    print("✅ Test 2 Passed: Trace preservation (column sums) correctly maintained.")


# ==============================================================================
# TEST 3: DEPHASING RATE UPDATE
# ==============================================================================
def test_dephasing_correction():
    """
    Verifies that coherence diagonal elements (i≠j) are updated by:
    ΔΓ_{ij} = 0.5 * (Δγ_i_out + Δγ_j_out)
    which corresponds to the standard Lindblad/Redfield dephasing rule.
    """
    N = 3
    energies = torch.tensor([0.0, 1.0, 2.5])
    T = torch.tensor(1.5)
    corrector = ThermalBalanceCorrector(energies, thermal_balance_mode="symmetric")
    
    torch.manual_seed(7)
    k_old = torch.rand(N, N) * 0.25
    k_old = 0.5 * (k_old + k_old.T)
    k_old.fill_diagonal_(0.0)
    
    N2 = N * N
    superop = torch.zeros(N2, N2)
    pop_idx = torch.arange(N) * (N + 1)
    superop[pop_idx[:, None], pop_idx[None, :]] = k_old
    # Start with zero dephasing to isolate the correction term
    superop[torch.arange(N2), torch.arange(N2)] = 0.0
    
    transformed = corrector.apply_superoperator_transform(T, superop)
    
    # Compute out-rates before and after
    gamma_out_old = k_old.sum(dim=-2)
    gamma_out_new = transformed[pop_idx[:, None], pop_idx[None, :]].sum(dim=-2)
    
    # Theoretical correction for ALL diagonal elements
    delta_gamma_out = gamma_out_new - gamma_out_old
    expected_delta_diag = 0.5 * (delta_gamma_out.unsqueeze(-1) + delta_gamma_out.unsqueeze(-2)).flatten()
    
    # Actual change in superoperator diagonal
    actual_delta_diag = transformed.diag() - superop.diag()
    
    # Mask population diagonals (we already tested them in Test 2)
    coherence_mask = torch.ones(N2, dtype=torch.bool)
    coherence_mask[pop_idx] = False
    
    assert torch.allclose(actual_delta_diag[coherence_mask], expected_delta_diag[coherence_mask], rtol=1e-5, atol=1e-8), \
        f"Dephasing correction mismatch. Max error: {(actual_delta_diag - expected_delta_diag).abs().max()}"
    print("✅ Test 3 Passed: Dephasing rates updated according to ½(Δγ_i + Δγ_j) rule.")

# ==============================================================================
# RUN ALL TESTS
# ==============================================================================
if __name__ == "__main__":
    test_detailed_balance()
    test_trace_preservation()
    test_dephasing_correction()
    print("\n🎉 All thermal balance tests passed successfully!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Test 1 Passed: Detailed balance correctly enforced.
✅ Test 2 Passed: Trace preservation (column sums) correctly maintained.
✅ Test 3 Passed: Dephasing rates updated according to ½(Δγ_i + Δγ_j) rule.

🎉 All thermal balance tests passed successfully!
